# 📓 Notebook 0｜維度詛咒與 Peaking 現象（站 0–1）

> 對應講義 Part 0–1。核心問題：**特徵是不是越多越好？**
> 答案：不是。樣本數 N 有限時，特徵數 l 超過臨界點，錯誤率反而上升——這就是 Peaking 現象。

In [ ]:
# 先跑這個：確認環境
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
print('numpy', np.__version__)
print('環境就緒 ✓')

### 課本例子：兩類高斯，均值 $\pm\boldsymbol{\mu}$

課本 5.3 用一個漂亮的例子說明 Peaking：兩類都是高斯、共變異數 $\Sigma=I$，
均值分別是 $+\boldsymbol{\mu}$ 與 $-\boldsymbol{\mu}$，其中

$$\boldsymbol{\mu}=\left[1,\frac{1}{\sqrt2},\frac{1}{\sqrt3},\dots,\frac{1}{\sqrt l}\right]^T$$

**已知均值**時，錯誤率 $P_e=\int_{b_l}^{\infty}\frac{1}{\sqrt{2\pi}}e^{-z^2/2}dz$，
其中 $b_l=\sqrt{\sum_{i=1}^l \frac1i}$。因為 $\sum 1/i\to\infty$，特徵越多錯誤率越低。

In [ ]:
# 已知均值：錯誤率隨 l 單調下降（特徵越多越好）
def pe_known(l):
    b = np.sqrt(np.sum(1.0 / np.arange(1, l + 1)))
    return 1 - norm.cdf(b)   # P(z > b)

ls = np.arange(1, 41)
pe = [pe_known(l) for l in ls]
plt.figure(figsize=(7, 4))
plt.plot(ls, pe, 'o-', color='#0ea5e9')
plt.xlabel('特徵數 l'); plt.ylabel('錯誤率 P_e')
plt.title('已知均值：特徵越多，錯誤率越低（單調下降）')
plt.grid(alpha=.3); plt.show()
print('l=1 錯誤率:', round(pe_known(1), 4), ' l=40 錯誤率:', round(pe_known(40), 6))

### 但均值要「估」時，故事反轉

實務上均值 $\boldsymbol{\mu}$ 要從 N 個訓練樣本估出來。課本證明此時
$b_l=\frac{E[z]}{\sigma_z}\to 0$，錯誤率衝向 **0.5（亂猜）**。

我們用蒙地卡羅重現：固定 N，逐步增加 l，看「測試錯誤率」先降後升。

In [ ]:
# 蒙地卡羅重現 Peaking：均值要估，錯誤率先降後升
def peaking_error(N, l, n_test=2000, seed=0):
    rng = np.random.default_rng(seed)
    mu = np.array([1.0 / np.sqrt(i) for i in range(1, l + 1)])
    # 訓練：兩類各 N/2 個樣本，估均值
    X1 = rng.normal(mu, 1.0, (N // 2, l)); X2 = rng.normal(-mu, 1.0, (N // 2, l))
    mu_hat = (X1.mean(0) - X2.mean(0)) / 2   # 估出的 mu
    # 測試：用估出的 mu 做最小距離分類
    T1 = rng.normal(mu, 1.0, (n_test, l)); T2 = rng.normal(-mu, 1.0, (n_test, l))
    def err(T, label):
        d1 = ((T - mu_hat) ** 2).sum(1); d2 = ((T + mu_hat) ** 2).sum(1)
        return (np.argmin(np.c_[d1, d2], 1) != label).mean()
    return (err(T1, 0) + err(T2, 1)) / 2

for N in [20, 100]:
    ls = np.arange(1, 31)
    errs = [peaking_error(N, l) for l in ls]
    plt.plot(ls, errs, 'o-', label=f'N={N}')
plt.xlabel('特徵數 l'); plt.ylabel('測試錯誤率')
plt.title('Peaking 現象：錯誤率先降後升，最低點隨 N 右移')
plt.legend(); plt.grid(alpha=.3); plt.show()
print('→ 樣本 N 越大，能撐的特徵越多（最低點越靠右）')

### 課本結論

- 最低點約在 $l=N/\alpha$，$\alpha$ 通常 2～10。
- **口訣**：「樣本少，特徵就要少」。

# ✏️ 練習
把上面的 `N` 改成 500，看最低點是否更靠右、錯誤率是否更低。